# EKD Paper Student ResNet-20 - Fixed Teacher Eval Compatible

Train the EKD ResNet-20 student required by:
Xiang, Gao, Xu. "Evidential Knowledge Distillation", ICCV 2025.

This file is standalone and Kaggle-ready. It follows the official EKD
CIFAR-100 ResNet-110 -> ResNet-20 config:

    configs/cifar100/ekd/resnet110_resnet20.yaml

Important: the teacher checkpoint must be an evidential teacher trained by
ekd_paper_teacher_resnet110.py or the official EKD teacher training code.
This is not the same as a normal softmax CE teacher.

Official EKD settings for this pair:
    WARMUP = 30
    STUDENT.LAMB = -0.86
    TEACHER.LAMB = -0.86
    CE_WEIGHT = 1.0
    FIRST_WEIGHT = 4.5
    SECOND_WEIGHT = 1.0
    EPOCHS = 240
    BATCH_SIZE = 64
    LR = 0.05
    LR_DECAY_STAGES = [150, 180, 210]

Outputs:
    /kaggle/working/kd_cifar100_artifacts/ekd_paper_student_resnet20/


## Imports and Configuration


In [1]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_INPUT = Path("/kaggle/input")
BASE_OUTPUT_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "outputs"

CONFIG = {
    "paper": "Evidential Knowledge Distillation, ICCV 2025",
    "official_config": "configs/cifar100/ekd/resnet110_resnet20.yaml",
    "method": "EKD",
    "model": "resnet20",
    "teacher_model": "resnet110",
    "dataset": "CIFAR100",
    "num_classes": 100,
    "epochs": 240,
    "batch_size": 64,
    "test_batch_size": 64,
    "optimizer": "SGD",
    "learning_rate": 0.05,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "scheduler": "MultiStepLR",
    "milestones": [150, 180, 210],
    "gamma": 0.1,
    "warmup": 30,
    "student_lamb": -0.86,
    "teacher_lamb": -0.86,
    "ce_weight": 1.0,
    "first_weight": 4.5,
    "second_weight": 1.0,
    "loss": "CE + EKD first-order + EKD second-order",
    "seed": 42,
    "num_workers": 2,
    "teacher_checkpoint_filename": "ekd_paper_teacher_resnet110_best.pth",
    "fixed_eval_teacher_checkpoint_filename": "ekd_paper_teacher_resnet110_best.pth",
    "teacher_input_dir": "/kaggle/input/models/zaidiftikhar/ekd-paper-teacher-resnet110-fixed-eval/pytorch/default/1",
    "fixed_eval_teacher_input_dir": "/kaggle/input/models/zaidiftikhar/ekd-paper-teacher-resnet110-fixed-eval/pytorch/default/1",
    "output_root": str(BASE_OUTPUT_DIR / "kd_cifar100_artifacts"),
    "stage_dir": "ekd_paper_student_resnet20_fixed_eval",
}


## Reproducibility


In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


## Model Building Block Helper


In [3]:
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(
        in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False
    )


## Class: `BasicBlock`


In [4]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, is_last=False):
        super().__init__()
        self.is_last = is_last
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out = out + residual
        preact = out
        out = F.relu(out, inplace=True)
        if self.is_last:
            return out, preact
        return out


## Class: `CIFARResNet`


In [5]:
class CIFARResNet(nn.Module):
    def __init__(self, depth, num_filters, num_classes=100):
        super().__init__()
        assert (depth - 2) % 6 == 0
        n = (depth - 2) // 6
        self.inplanes = num_filters[0]
        self.conv1 = nn.Conv2d(3, num_filters[0], kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_filters[0])
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(BasicBlock, num_filters[1], n)
        self.layer2 = self._make_layer(BasicBlock, num_filters[2], n, stride=2)
        self.layer3 = self._make_layer(BasicBlock, num_filters[3], n, stride=2)
        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(num_filters[3], num_classes)
        self.stage_channels = num_filters
        self._initialize_weights()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.inplanes,
                    planes * block.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(planes * block.expansion),
            )
        layers = [block(self.inplanes, planes, stride, downsample, is_last=(blocks == 1))]
        self.inplanes = planes * block.expansion
        for idx in range(1, blocks):
            layers.append(block(self.inplanes, planes, is_last=(idx == blocks - 1)))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        f0 = x
        x, f1_pre = self.layer1(x)
        f1 = x
        x, f2_pre = self.layer2(x)
        f2 = x
        x, f3_pre = self.layer3(x)
        f3 = x
        x = self.avgpool(x)
        pooled = x.reshape(x.size(0), -1)
        logits = self.fc(pooled)
        feats = {
            "feats": [f0, f1, f2, f3],
            "preact_feats": [f0, f1_pre, f2_pre, f3_pre],
            "pooled_feat": pooled,
        }
        return logits, feats


## Model Factories


In [6]:
def resnet20(num_classes=100):
    return CIFARResNet(20, [16, 16, 32, 64], num_classes=num_classes)

def resnet110(num_classes=100):
    return CIFARResNet(110, [16, 16, 32, 64], num_classes=num_classes)


## CIFAR-100 Data Loaders


In [7]:
def build_loaders():
    mean = (0.5071, 0.4867, 0.4408)
    std = (0.2675, 0.2565, 0.2761)
    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    test_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean, std)]
    )
    data_root = KAGGLE_WORKING / "data" if KAGGLE_WORKING.exists() else Path("./data")
    train_set = datasets.CIFAR100(
        root=str(data_root), train=True, download=True, transform=train_transform
    )
    test_set = datasets.CIFAR100(
        root=str(data_root), train=False, download=True, transform=test_transform
    )
    train_loader = DataLoader(
        train_set,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=CONFIG["test_batch_size"],
        shuffle=False,
        num_workers=1,
        pin_memory=torch.cuda.is_available(),
    )
    return train_set, test_set, train_loader, test_loader


## Teacher Checkpoint Discovery


In [8]:
def find_teacher_checkpoint():
    filenames = [
        CONFIG["fixed_eval_teacher_checkpoint_filename"],
        CONFIG["teacher_checkpoint_filename"],
    ]
    roots = []
    for input_key in ["fixed_eval_teacher_input_dir", "teacher_input_dir"]:
        explicit = Path(CONFIG[input_key])
        if explicit.exists():
            roots.append(explicit)
    if KAGGLE_INPUT.exists():
        roots.extend(sorted(KAGGLE_INPUT.glob("*")))
    roots.extend(
        [
            Path(CONFIG["output_root"]) / "ekd_paper_teacher_resnet110",
            Path.cwd() / "kd_cifar100_artifacts" / "ekd_paper_teacher_resnet110",
            Path.cwd(),
        ]
    )
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for filename in filenames:
            candidates.extend(
                [
                    root / filename,
                    root / "checkpoints" / filename,
                    root / "ekd_paper_teacher_resnet110" / "checkpoints" / filename,
                    root / "ekd_paper_teacher_resnet110_fixed_eval" / "checkpoints" / filename,
                ]
            )
            candidates.extend(root.rglob(filename))
    seen = set()
    unique = []
    for path in candidates:
        if path.exists():
            resolved = path.resolve()
            if resolved not in seen:
                seen.add(resolved)
                unique.append(path)
    if not unique:
        raise FileNotFoundError(
            f"Could not find any teacher checkpoint from {filenames}. Train the fixed-eval evidential teacher first or attach it as a Kaggle model input."
        )
    return unique[0]


## Teacher Loading


In [9]:
def load_teacher_model(path, device):
    teacher = resnet110(num_classes=CONFIG["num_classes"]).to(device)
    checkpoint = torch.load(path, map_location=device)
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "student" in checkpoint:
        state_dict = checkpoint["student"]
    elif "model" in checkpoint and isinstance(checkpoint["model"], dict):
        state_dict = checkpoint["model"]
    else:
        state_dict = checkpoint
    teacher.load_state_dict(state_dict)
    teacher.eval()
    for param in teacher.parameters():
        param.requires_grad = False
    return teacher, checkpoint


## EKD Second-Order Loss


In [10]:
def compute_ekd_loss(alpha_student, alpha_teacher):
    student_strength = torch.sum(alpha_student, dim=1)
    teacher_strength = torch.sum(alpha_teacher, dim=1)
    teacher_strength_keep = torch.sum(alpha_teacher, dim=1, keepdim=True)
    loss_term1 = torch.lgamma(teacher_strength) - torch.lgamma(student_strength)
    loss_term2 = -torch.sum(torch.lgamma(alpha_teacher) - torch.lgamma(alpha_student), dim=1)
    loss_term3 = torch.sum(
        (alpha_teacher - alpha_student)
        * (torch.digamma(alpha_teacher) - torch.digamma(teacher_strength_keep)),
        dim=1,
    )
    return (loss_term1 + loss_term2 + loss_term3).mean()


## EKD First-Order Loss


In [11]:
def evidential_first_order_loss(alpha_student, alpha_teacher):
    student_strength = torch.sum(alpha_student, dim=1, keepdim=True)
    teacher_strength = torch.sum(alpha_teacher, dim=1, keepdim=True)
    log_pred_student = torch.log(alpha_student / student_strength)
    pred_teacher = alpha_teacher / teacher_strength
    return F.kl_div(log_pred_student, pred_teacher, reduction="none").sum(1).mean()


## EKD Student Objective


In [12]:
def ekd_total_loss(student_logits, teacher_logits, labels, epoch):
    lamb_student = torch.tensor(CONFIG["student_lamb"], device=student_logits.device)
    lamb_teacher = torch.tensor(CONFIG["teacher_lamb"], device=student_logits.device)
    labels_1hot = torch.zeros_like(student_logits).scatter_(-1, labels.unsqueeze(-1), 1)

    evidence_student = torch.exp(student_logits)
    alpha_student = evidence_student + torch.exp(lamb_student)
    strength_student = torch.sum(alpha_student, dim=-1, keepdim=True)
    loss_ce = torch.sum(
        labels_1hot * (torch.digamma(strength_student) - torch.digamma(alpha_student)),
        dim=-1,
    ).mean()

    evidence_student = torch.exp(student_logits)
    evidence_teacher = torch.exp(teacher_logits)
    alpha_student_first = evidence_student + torch.exp(lamb_student)
    alpha_teacher_first = evidence_teacher + torch.exp(lamb_teacher)
    loss_first = CONFIG["first_weight"] * evidential_first_order_loss(
        alpha_student_first, alpha_teacher_first
    )

    alpha_student_second = torch.log1p(evidence_student) + torch.exp(lamb_student)
    alpha_teacher_second = torch.log1p(evidence_teacher) + torch.exp(lamb_teacher)
    warmup_weight = min(epoch / CONFIG["warmup"], 1.0)
    loss_second = (
        warmup_weight
        * CONFIG["second_weight"]
        * compute_ekd_loss(alpha_student_second, alpha_teacher_second)
    )

    total = CONFIG["ce_weight"] * loss_ce + loss_first + loss_second
    return total, loss_ce.detach(), loss_first.detach(), loss_second.detach(), warmup_weight


## Metrics


In [13]:
def topk_accuracy(outputs, labels, topk=(1, 5)):
    maxk = max(topk)
    _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)
    pred = pred.t()
    correct = pred.eq(labels.reshape(1, -1).expand_as(pred))
    results = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0)
        results.append((correct_k * (100.0 / labels.size(0))).item())
    return results


## Evaluation


In [14]:
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total = 0
    criterion = nn.CrossEntropyLoss()
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits, _ = model(images)
        loss = criterion(logits, labels)
        top1, top5 = topk_accuracy(logits, labels, topk=(1, 5))
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_top1 += top1 * batch_size
        total_top5 += top5 * batch_size
        total += batch_size
    return total_loss / total, total_top1 / total, total_top5 / total


@torch.no_grad()
def evaluate_teacher_raw_logits(teacher_model, loader, device):
    teacher_model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total = 0
    criterion = nn.CrossEntropyLoss()
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits, _ = teacher_model(images)
        loss = criterion(logits, labels)
        top1, top5 = topk_accuracy(logits, labels, topk=(1, 5))
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_top1 += top1 * batch_size
        total_top5 += top5 * batch_size
        total += batch_size
    return total_loss / total, total_top1 / total, total_top5 / total


## Checkpointing


In [15]:
def save_checkpoint(path, epoch, model, optimizer, scheduler, best_acc, config, train_set):
    checkpoint = {
        "epoch": epoch,
        "model_name": config["model"],
        "teacher_model": config["teacher_model"],
        "method": config["method"],
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_acc": best_acc,
        "config": config,
        "class_to_idx": train_set.class_to_idx,
        "classes": train_set.classes,
    }
    torch.save(checkpoint, path)


## Plotting


In [16]:
def plot_curves(log_df, plot_dir):
    plt.figure(figsize=(8, 5))
    plt.plot(log_df["epoch"], log_df["train_acc"], label="Train Accuracy")
    plt.plot(log_df["epoch"], log_df["test_acc"], label="Test Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("EKD Paper Student Accuracy")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / "ekd_paper_student_accuracy_curve.png", dpi=200)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(log_df["epoch"], log_df["train_loss"], label="Total Loss")
    plt.plot(log_df["epoch"], log_df["loss_ce"], label="Evidential CE")
    plt.plot(log_df["epoch"], log_df["loss_first"], label="First-order EKD")
    plt.plot(log_df["epoch"], log_df["loss_second"], label="Second-order EKD")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("EKD Paper Student Loss Components")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_dir / "ekd_paper_student_loss_curve.png", dpi=200)
    plt.close()


## Training Pipeline


In [17]:
def main():
    set_seed(CONFIG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    stage_root = Path(CONFIG["output_root"]) / CONFIG["stage_dir"]
    checkpoint_dir = stage_root / "checkpoints"
    log_dir = stage_root / "logs"
    config_dir = stage_root / "config"
    plot_dir = stage_root / "plots"
    for directory in [checkpoint_dir, log_dir, config_dir, plot_dir]:
        directory.mkdir(parents=True, exist_ok=True)
    with open(config_dir / "ekd_paper_student_resnet20_config.json", "w") as f:
        json.dump(CONFIG, f, indent=4)

    train_set, _, train_loader, test_loader = build_loaders()
    teacher_path = find_teacher_checkpoint()
    teacher, teacher_ckpt = load_teacher_model(teacher_path, device)
    print(f"Loaded evidential teacher: {teacher_path}")
    print(f"Teacher checkpoint best_acc: {teacher_ckpt.get('best_acc') if isinstance(teacher_ckpt, dict) else None}")
    teacher_ce_loss, teacher_acc, teacher_top5 = evaluate_teacher_raw_logits(teacher, test_loader, device)
    print(f"Teacher raw-logit CE: {teacher_ce_loss:.4f} | top1: {teacher_acc:.2f}% | top5: {teacher_top5:.2f}%")

    student = resnet20(num_classes=CONFIG["num_classes"]).to(device)
    optimizer = optim.SGD(
        student.parameters(),
        lr=CONFIG["learning_rate"],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"],
    )
    scheduler = MultiStepLR(
        optimizer, milestones=CONFIG["milestones"], gamma=CONFIG["gamma"]
    )

    best_acc = -1.0
    records = []
    start_time = time.time()
    best_path = checkpoint_dir / "ekd_paper_student_resnet20_best.pth"
    last_path = checkpoint_dir / "ekd_paper_student_resnet20_last.pth"
    log_path = log_dir / "ekd_paper_student_training_log.csv"

    for epoch in range(1, CONFIG["epochs"] + 1):
        epoch_start = time.time()
        student.train()
        total_loss = 0.0
        total_ce = 0.0
        total_first = 0.0
        total_second = 0.0
        total_top1 = 0.0
        total_top5 = 0.0
        total = 0
        current_lr = optimizer.param_groups[0]["lr"]

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True).float()
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            student_logits, _ = student(images)
            with torch.no_grad():
                teacher_logits, _ = teacher(images)
            loss, loss_ce, loss_first, loss_second, warmup_weight = ekd_total_loss(
                student_logits, teacher_logits, labels, epoch
            )
            loss.backward()
            optimizer.step()

            top1, top5 = topk_accuracy(student_logits, labels, topk=(1, 5))
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_ce += loss_ce.item() * batch_size
            total_first += loss_first.item() * batch_size
            total_second += loss_second.item() * batch_size
            total_top1 += top1 * batch_size
            total_top5 += top5 * batch_size
            total += batch_size

        train_loss = total_loss / total
        train_acc = total_top1 / total
        train_acc_top5 = total_top5 / total
        test_loss, test_acc, test_acc_top5 = evaluate(student, test_loader, device)
        scheduler.step()

        if test_acc >= best_acc:
            best_acc = test_acc
            save_checkpoint(best_path, epoch, student, optimizer, scheduler, best_acc, CONFIG, train_set)
        save_checkpoint(last_path, epoch, student, optimizer, scheduler, best_acc, CONFIG, train_set)

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "warmup_weight": warmup_weight,
            "train_loss": train_loss,
            "loss_ce": total_ce / total,
            "loss_first": total_first / total,
            "loss_second": total_second / total,
            "train_acc": train_acc,
            "train_acc_top5": train_acc_top5,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "test_acc_top5": test_acc_top5,
            "best_test_acc": best_acc,
            "epoch_time_sec": time.time() - epoch_start,
        }
        records.append(record)
        pd.DataFrame(records).to_csv(log_path, index=False)
        print(
            f"Epoch {epoch:03d}/{CONFIG['epochs']} | lr {current_lr:.5f} | "
            f"train {train_loss:.4f}, acc {train_acc:.2f}% | "
            f"test {test_loss:.4f}, acc {test_acc:.2f}% | best {best_acc:.2f}% | "
            f"first {record['loss_first']:.4f} second {record['loss_second']:.4f}"
        )

    log_df = pd.DataFrame(records)
    plot_curves(log_df, plot_dir)

    summary = {
        "method": CONFIG["method"],
        "model": "ResNet20",
        "teacher": "ResNet110 Evidential Teacher",
        "dataset": "CIFAR100",
        "best_test_acc": best_acc,
        "final_test_acc": float(log_df.iloc[-1]["test_acc"]),
        "final_train_acc": float(log_df.iloc[-1]["train_acc"]),
        "params": sum(p.numel() for p in student.parameters()),
        "epochs": CONFIG["epochs"],
        "batch_size": CONFIG["batch_size"],
        "optimizer": CONFIG["optimizer"],
        "scheduler": CONFIG["scheduler"],
        "seed": CONFIG["seed"],
        "warmup": CONFIG["warmup"],
        "student_lamb": CONFIG["student_lamb"],
        "teacher_lamb": CONFIG["teacher_lamb"],
        "ce_weight": CONFIG["ce_weight"],
        "first_weight": CONFIG["first_weight"],
        "second_weight": CONFIG["second_weight"],
        "teacher_checkpoint": str(teacher_path),
        "teacher_raw_logit_ce": teacher_ce_loss,
        "teacher_top1_acc": teacher_acc,
        "teacher_top5_acc": teacher_top5,
        "checkpoint_path": str(best_path),
        "last_checkpoint_path": str(last_path),
        "training_time_seconds": time.time() - start_time,
    }
    summary_path = stage_root / "ekd_paper_student_resnet20_summary.json"
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=4)
    print(json.dumps(summary, indent=4))


## Run Training

Run this cell to start the full paper-matched training job.


In [18]:
main()


Using device: cuda
GPU: Tesla T4


100%|██████████| 169M/169M [00:01<00:00, 95.0MB/s] 


Loaded evidential teacher: /kaggle/input/models/zaidiftikhar/ekd-paper-teacher-resnet110-fixed-eval/pytorch/default/1/ekd_paper_teacher_resnet110_best.pth
Teacher checkpoint best_acc: 73.69
Teacher raw-logit CE: 1.1442 | top1: 73.69% | top5: 92.87%
Epoch 001/240 | lr 0.05000 | train 23.9354, acc 7.21% | test 3.6859, acc 12.93% | best 12.93% | first 17.4295 second 2.1998
Epoch 002/240 | lr 0.05000 | train 22.0682, acc 18.41% | test 3.2494, acc 21.55% | best 21.55% | first 14.4785 second 4.0469
Epoch 003/240 | lr 0.05000 | train 21.0257, acc 27.11% | test 2.8175, acc 29.54% | best 29.54% | first 12.3904 second 5.6121
Epoch 004/240 | lr 0.05000 | train 20.2681, acc 35.13% | test 2.5982, acc 34.07% | best 34.07% | first 10.7286 second 6.9182
Epoch 005/240 | lr 0.05000 | train 20.0341, acc 40.08% | test 2.3763, acc 38.13% | best 38.13% | first 9.6445 second 8.0326
Epoch 006/240 | lr 0.05000 | train 20.1572, acc 43.91% | test 2.2671, acc 40.15% | best 40.15% | first 8.8882 second 9.0925
Epoc

## Zip Results

Run this after training to package the notebook outputs.


In [19]:
import shutil
from pathlib import Path

stage_root = Path(CONFIG["output_root"]) / "ekd_paper_student_resnet20"
zip_base = BASE_OUTPUT_DIR / "ekd_paper_student_resnet20_fixed_eval_artifacts"

if not stage_root.exists():
    print("Artifact directory does not exist yet:", stage_root)
    print("Run the training cell above before zipping outputs.")
else:
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=stage_root.parent, base_dir=stage_root.name)
    print("Created zip file:")
    print(zip_path)


Artifact directory does not exist yet: /kaggle/working/kd_cifar100_artifacts/ekd_paper_student_resnet20
Run the training cell above before zipping outputs.


In [20]:
import shutil
from pathlib import Path

stage_root = Path("/kaggle/working/kd_cifar100_artifacts/ekd_paper_student_resnet20_fixed_eval")
zip_base = Path("/kaggle/working/kd_cifar100_artifacts/ekd_paper_student_resnet20_fixed_eval_artifacts")

if not stage_root.exists():
    print("Artifact directory does not exist yet:", stage_root)
    print("Run the training/evaluation cells before zipping outputs.")
else:
    zip_path = shutil.make_archive(
        str(zip_base),
        "zip",
        root_dir=stage_root.parent,
        base_dir=stage_root.name
    )
    print("Created zip file:")
    print(zip_path)

Created zip file:
/kaggle/working/kd_cifar100_artifacts/ekd_paper_student_resnet20_fixed_eval_artifacts.zip
